In [1]:
import sys
sys.path.append('../../bats_transformer')

In [2]:
from argparse import ArgumentParser
import torch
import numpy as np

import pytorch_lightning as pl
import spacetimeformer as stf
import pandas as pd
import scipy.stats as stats

from pytorch_lightning.loggers import WandbLogger
from data import preprocess
import time
import tqdm
from itertools import chain
from data.bats_dataset import *
from pytorch_lightning.callbacks import LearningRateMonitor

from itertools import chain
from utils import *

In [3]:
outputs_1_file = "../../bats_transformer/outputs/july_daytime_original_10_27/model_31_losses.log"
outputs_2_file = "../../bats_transformer/outputs/july_daytime_chunked_10_27/model_31_losses.log"

In [4]:
outputs_1 = pd.read_csv(outputs_1_file)
outputs_2 = pd.read_csv(outputs_2_file)
outputs_1

,Unnamed: 0,TimeIndex,TimeInFile,PrecedingIntrvl,HiFreq,Bndwdth,FreqMaxPwr,PrcntMaxAmpDur,FreqKnee,PrcntKneeDur,...,Amp4thQrtl,1st10kHzSlp,1st5to15kHzSlp,1st10kHzExp,1st5to15kHzExp,AmpK@start,MaxSegLngth,file_id,chirp_idx,model_id
0,0,1.718572,0.019925,0.020794,0.005719,0.302850,0.109781,0.349674,0.006319,0.562072,...,0.011351,0.114464,0.000897,0.001307,0.000576,0.423525,0.000025,3491.0,48.0,models/july_daytime_original_10_27/model_31
1,1,1.718572,0.000303,0.006761,0.000993,0.007707,0.001641,0.021465,0.039666,0.020814,...,0.073436,0.091797,0.001038,0.003369,0.008295,0.386010,0.000778,3491.0,47.0,models/july_daytime_original_10_27/model_31
2,2,1.718572,0.017893,0.097069,1.365424,2.962165,0.115368,1.254639,0.006135,0.788661,...,0.041993,0.000118,0.424397,0.001046,0.001960,7.037145,0.000511,3491.0,46.0,models/july_daytime_original_10_27/model_31
3,3,1.718572,0.034425,0.011177,0.011193,0.031790,0.089440,0.052740,0.040149,0.002312,...,0.002159,0.116796,0.004064,0.002655,0.000071,0.018653,0.006283,3491.0,45.0,models/july_daytime_original_10_27/model_31
4,4,1.718572,0.001679,0.051215,0.021635,0.013772,0.293371,0.006939,0.131957,0.026204,...,1.235373,0.123037,0.073059,0.000073,0.003164,0.050194,0.005821,3491.0,44.0,models/july_daytime_original_10_27/model_31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5302,5302,1.718572,0.548345,0.005301,0.309374,0.795308,0.003503,0.468036,0.032421,0.286268,...,2.596288,0.194213,0.038158,0.006618,0.010699,1.689896,0.011797,3608.0,10.0,models/july_daytime_original_10_27/model_31
5303,5303,1.718572,0.108942,0.075465,0.000204,0.040454,0.010517,0.017490,0.052559,0.095353,...,0.105475,0.026579,0.154736,0.039242,0.008202,0.068784,0.011537,3608.0,9.0,models/july_daytime_original_10_27/model_31
5304,5304,1.718572,0.311845,0.041688,0.003681,0.263558,0.951904,0.001390,0.014278,1.303001,...,0.919697,0.179421,0.333609,0.036892,0.039444,0.048988,0.032340,3608.0,8.0,models/july_daytime_original_10_27/model_31
5305,5305,1.718572,0.151231,0.148787,0.142843,0.098942,0.031000,0.382204,0.079179,0.147527,...,0.108511,0.007857,0.220386,0.012327,0.003062,0.163638,0.034756,3608.0,7.0,models/july_daytime_original_10_27/model_31


In [5]:
def checkFeatureDistributionDifference(feature):
    losses_1 = outputs_1[feature].values
    losses_2 = outputs_2[feature].values
    # align and drop NaNs
    mask = ~np.isnan(losses_1) & ~np.isnan(losses_2)
    l1 = losses_1[mask]
    l2 = losses_2[mask]
    diffs = l2 - l1

    # print(f"n={len(diffs)}, mean_diff={diffs.mean():.6g}, std_diff={diffs.std(ddof=1):.6g}")

    # check normality of differences (Shapiro-Wilk valid for 3..5000 samples)
    if 3 <= len(diffs) <= 5000:
        sh = stats.shapiro(diffs)
        # print(f"Shapiro-Wilk: W={sh.statistic:.4f}, p={sh.pvalue:.4g}")
    else:
        print("Skipping Shapiro-Wilk (sample size outside 3..5000)")

    # paired t-test (two-sided returned by scipy) -> convert to one-sided "less" (mean(diffs) < 0)
    t_res = stats.ttest_ind(l2, l1, nan_policy="omit")
    t_stat, t_p_two = t_res.statistic, t_res.pvalue
    if np.isnan(t_stat):
        print("Paired t-test returned NaN")
    else:
        if t_stat < 0:
            t_p_one = t_p_two / 2
        else:
            t_p_one = 1 - t_p_two / 2
        # print(f"Paired t-test: t={t_stat:.4f}, two-sided p={t_p_two:.4g}, one-sided p (losses_2 < losses_1)={t_p_one:.4g}")

    # Wilcoxon signed-rank test (non-parametric paired)
    try:
        w_res = stats.wilcoxon(diffs)
        w_stat, w_p_two = w_res.statistic, w_res.pvalue
        # convert two-sided to one-sided
        # use sign of median(diff) to decide direction
        if np.median(diffs) < 0:
            w_p_one = w_p_two / 2
        else:
            w_p_one = 1 - w_p_two / 2
        # print(f"Wilcoxon signed-rank: W={w_stat:.4f}, two-sided p={w_p_two:.4g}, one-sided p (losses_2 < losses_1)={w_p_one:.4g}")
    except Exception as e:
        print("Wilcoxon test failed:", e)

    # paired effect size (Cohen's d for paired samples)
    mean_diff = diffs.mean()
    sd_diff = diffs.std(ddof=1)
    cohen_d = mean_diff / sd_diff if sd_diff > 0 else np.nan
    # print(f"Cohen's d (paired) = {cohen_d:.4f}")
    return {"diff": mean_diff,
            "shapiro_wilk": sh if 3 <= len(diffs) <= 5000 else None,
            "paired_t_test": t_p_one,
            "wilcoxon": w_p_one if 'w_res' in locals() else None,
            "cohen_d": cohen_d}

In [6]:
predicted_columns = outputs_1.columns.tolist()[2:]
predicted_columns = predicted_columns[:-3]

In [7]:
# outputs_1, outputs_2 = outputs_2, outputs_1

In [8]:
count_significant = 0
for column in predicted_columns:
    diff_stats = checkFeatureDistributionDifference(column)
    print(diff_stats)
    if diff_stats["paired_t_test"] > 0.05 or diff_stats["wilcoxon"] > 0.05:
        print(f"No significant difference in {column}, t-test p={diff_stats['paired_t_test']:.4g}, Wilcoxon p={diff_stats['wilcoxon']:.4g}")
    else:
        count_significant += 1
print(f"Out of {len(predicted_columns)} features, {count_significant} show significant difference between models.")

ValueError: operands could not be broadcast together with shapes (5307,) (4567,) 